# 04 - Offline LLM Fraud Explanation Layer

This notebook prototypes the final explanation layer for the LLM-powered financial fraud detection project.

The machine learning pipeline is already complete:
- Final model: Random Forest
- Final feature version: Version B - Without ExtremeFlags
- Approximate best results: PR-AUC 0.816, ROC-AUC 0.986, precision 0.924, recall 0.768, F1 0.839
- SHAP explainability artifacts are saved in `outputs/plots/shap/`
- SHAP feature importance is saved in `outputs/shap_feature_importance.csv`

This notebook does **not** call OpenAI, LangChain, or any paid API. It uses offline Python functions and prompt templates to demonstrate how model outputs and SHAP values can be converted into human-readable fraud investigation explanations.

## 1. Why an LLM Explanation Layer Helps

Fraud models can produce strong risk scores, but investigators need more than a probability. They need a concise explanation of why a transaction was flagged, which signals drove the model decision, what caveats apply, and what action should be considered next.

In this prototype, SHAP provides model-grounded evidence. The explanation layer turns that evidence into analyst-friendly language. The generated text is grounded in model outputs and SHAP feature contributions; it should not invent reasons outside the available evidence.

## 2. Setup

This section defines paths and imports standard Python libraries only. No external LLM service is used.

In [ ]:
from pathlib import Path
import textwrap

import numpy as np
import pandas as pd

SHAP_IMPORTANCE_PATH = Path("outputs/shap_feature_importance.csv")
EXPLANATIONS_PATH = Path("outputs/llm_fraud_explanations.csv")
REPORT_PATH = Path("outputs/llm_explanation_report.md")

MODEL_NAME = "Random Forest"
FEATURE_VERSION = "Version B - Without ExtremeFlags"
MODEL_THRESHOLD = 0.50

EXPLANATIONS_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

print("Offline prototype initialized.")
print(f"Input SHAP importance: {SHAP_IMPORTANCE_PATH}")
print(f"CSV output: {EXPLANATIONS_PATH}")
print(f"Markdown report output: {REPORT_PATH}")

**Recruiter-friendly takeaway:** The layer is intentionally safe and reproducible. It demonstrates the product concept without requiring paid APIs, secret keys, or live model serving.

## 3. Load SHAP Feature Importance

The SHAP table ranks features by their average absolute impact on the fraud model. These values provide global evidence about which features tend to matter most across transactions.

In [ ]:
if not SHAP_IMPORTANCE_PATH.exists():
    raise FileNotFoundError(
        f"Missing {SHAP_IMPORTANCE_PATH}. Run 03_explainability_shap.ipynb first to create it."
    )

shap_importance = pd.read_csv(SHAP_IMPORTANCE_PATH)

required_cols = {"feature", "mean_abs_shap"}
missing_cols = required_cols - set(shap_importance.columns)
if missing_cols:
    raise ValueError(f"SHAP importance file is missing required columns: {missing_cols}")

shap_importance = shap_importance.sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
display(shap_importance.head(15))

**Recruiter-friendly takeaway:** The explanation layer starts from SHAP, not intuition. This keeps the narrative tied to the model's measured behavior.

## 4. Feature Description Guardrails

`V1` through `V28` are anonymized PCA transaction behavior components. Their real business meanings are not available in this dataset, so the explanation layer must not pretend to know what they represent.

For those features, the safest wording is that they are anonymized transaction behavior components that influenced the model score.

In [ ]:
def describe_feature(feature_name):
    """Return safe, non-invented feature descriptions for explanation text."""
    if feature_name.startswith("V") and feature_name[1:].isdigit():
        return f"{feature_name}, an anonymized PCA transaction behavior component"

    descriptions = {
        "LogAmount": "the log-transformed transaction amount",
        "IsZeroAmount": "whether the transaction amount was recorded as zero",
        "Day": "the transaction day derived from timestamp information",
        "HourSin": "the cyclical time-of-day sine feature",
        "HourCos": "the cyclical time-of-day cosine feature",
    }
    return descriptions.get(feature_name, f"{feature_name}, an engineered model feature")


for feature in shap_importance["feature"].head(10):
    print(f"{feature}: {describe_feature(feature)}")

**Recruiter-friendly takeaway:** This guardrail is important for trust. The system can explain model behavior without inventing business meanings for anonymized PCA fields.

## 5. Create Example Transaction Explanation Records

A production system would pass in per-transaction SHAP contributions from the trained model. This offline notebook creates realistic example records from the saved SHAP importance table so the explanation layer can be demonstrated without retraining the model or calling an API.

Each record includes:
- Prediction label
- Fraud probability
- Top positive SHAP-style drivers
- Top negative SHAP-style drivers
- Transaction amount or log amount when available
- Model decision threshold

In [ ]:
def build_driver_list(features, sign=1, scale=1.0):
    drivers = []
    for rank, row in enumerate(features.itertuples(index=False), start=1):
        shap_value = sign * float(row.mean_abs_shap) * scale / rank
        drivers.append({
            "feature": row.feature,
            "description": describe_feature(row.feature),
            "shap_value": shap_value,
        })
    return drivers


top_features = shap_importance.head(8)
positive_core = top_features.head(4)
negative_core = top_features.tail(4)

example_records = [
    {
        "transaction_id": "TXN_HIGH_RISK_001",
        "case_type": "high-risk fraud prediction",
        "prediction_label": "Fraud",
        "actual_label_for_demo": "Fraud",
        "fraud_probability": 0.962,
        "decision_threshold": MODEL_THRESHOLD,
        "transaction_amount": 245.31,
        "log_amount": 5.507,
        "top_positive_shap_features": build_driver_list(positive_core, sign=1, scale=1.20),
        "top_negative_shap_features": build_driver_list(negative_core, sign=-1, scale=0.45),
    },
    {
        "transaction_id": "TXN_HIGH_RISK_002",
        "case_type": "high-risk fraud prediction",
        "prediction_label": "Fraud",
        "actual_label_for_demo": "Fraud",
        "fraud_probability": 0.887,
        "decision_threshold": MODEL_THRESHOLD,
        "transaction_amount": 82.00,
        "log_amount": 4.419,
        "top_positive_shap_features": build_driver_list(top_features.iloc[[1, 2, 3, 4]], sign=1, scale=1.05),
        "top_negative_shap_features": build_driver_list(top_features.iloc[[5, 6, 7]], sign=-1, scale=0.35),
    },
    {
        "transaction_id": "TXN_FALSE_NEGATIVE_001",
        "case_type": "false negative",
        "prediction_label": "Legitimate",
        "actual_label_for_demo": "Fraud",
        "fraud_probability": 0.431,
        "decision_threshold": MODEL_THRESHOLD,
        "transaction_amount": 19.99,
        "log_amount": 3.044,
        "top_positive_shap_features": build_driver_list(top_features.iloc[[0, 2]], sign=1, scale=0.55),
        "top_negative_shap_features": build_driver_list(top_features.iloc[[3, 4, 5, 6]], sign=-1, scale=0.95),
    },
    {
        "transaction_id": "TXN_FALSE_NEGATIVE_002",
        "case_type": "false negative",
        "prediction_label": "Legitimate",
        "actual_label_for_demo": "Fraud",
        "fraud_probability": 0.277,
        "decision_threshold": MODEL_THRESHOLD,
        "transaction_amount": 0.00,
        "log_amount": 0.000,
        "top_positive_shap_features": build_driver_list(top_features.iloc[[1, 5]], sign=1, scale=0.35),
        "top_negative_shap_features": build_driver_list(top_features.iloc[[0, 3, 4, 7]], sign=-1, scale=0.85),
    },
    {
        "transaction_id": "TXN_LOW_RISK_001",
        "case_type": "legitimate low-risk transaction",
        "prediction_label": "Legitimate",
        "actual_label_for_demo": "Legitimate",
        "fraud_probability": 0.018,
        "decision_threshold": MODEL_THRESHOLD,
        "transaction_amount": 32.45,
        "log_amount": 3.509,
        "top_positive_shap_features": build_driver_list(top_features.iloc[[6]], sign=1, scale=0.10),
        "top_negative_shap_features": build_driver_list(top_features.iloc[[0, 1, 2, 3]], sign=-1, scale=1.10),
    },
    {
        "transaction_id": "TXN_LOW_RISK_002",
        "case_type": "legitimate low-risk transaction",
        "prediction_label": "Legitimate",
        "actual_label_for_demo": "Legitimate",
        "fraud_probability": 0.006,
        "decision_threshold": MODEL_THRESHOLD,
        "transaction_amount": 11.20,
        "log_amount": 2.501,
        "top_positive_shap_features": [],
        "top_negative_shap_features": build_driver_list(top_features.iloc[[1, 2, 4, 5]], sign=-1, scale=1.00),
    },
]

example_overview = pd.DataFrame([
    {
        "transaction_id": record["transaction_id"],
        "case_type": record["case_type"],
        "prediction_label": record["prediction_label"],
        "actual_label_for_demo": record["actual_label_for_demo"],
        "fraud_probability": record["fraud_probability"],
        "decision_threshold": record["decision_threshold"],
        "transaction_amount": record["transaction_amount"],
        "log_amount": record["log_amount"],
    }
    for record in example_records
])

display(example_overview)

**Recruiter-friendly takeaway:** These examples show the data contract the LLM layer would need in production: model score, threshold, and per-feature SHAP drivers.

## 6. Structured Explanation Prompt Template

A future LLM call should receive structured, bounded context. The prompt should instruct the model to use only the supplied prediction and SHAP evidence.

In [ ]:
PROMPT_TEMPLATE = """
You are a fraud investigation assistant. Write a concise explanation grounded only in the supplied model outputs and SHAP feature contributions.

Rules:
- Do not invent reasons that are not supported by SHAP values.
- Do not claim that anonymized PCA features V1-V28 have real-world business meanings.
- Refer to V1-V28 only as anonymized transaction behavior components.
- Include a risk summary, key model drivers, suggested investigation action, and confidence caveat.

Model context:
- Model: {model_name}
- Feature version: {feature_version}
- Decision threshold: {decision_threshold:.2f}

Transaction context:
- Transaction ID: {transaction_id}
- Prediction label: {prediction_label}
- Fraud probability: {fraud_probability:.3f}
- Transaction amount: {transaction_amount}
- Log amount: {log_amount}
- Positive SHAP drivers: {positive_drivers}
- Negative SHAP drivers: {negative_drivers}
""".strip()


def format_driver_compact(driver):
    return f"{driver['feature']} ({driver['shap_value']:+.4f})"


sample_prompt = PROMPT_TEMPLATE.format(
    model_name=MODEL_NAME,
    feature_version=FEATURE_VERSION,
    decision_threshold=MODEL_THRESHOLD,
    transaction_id=example_records[0]["transaction_id"],
    prediction_label=example_records[0]["prediction_label"],
    fraud_probability=example_records[0]["fraud_probability"],
    transaction_amount=example_records[0]["transaction_amount"],
    log_amount=example_records[0]["log_amount"],
    positive_drivers=", ".join(format_driver_compact(d) for d in example_records[0]["top_positive_shap_features"]),
    negative_drivers=", ".join(format_driver_compact(d) for d in example_records[0]["top_negative_shap_features"]),
)

print(sample_prompt)

**Recruiter-friendly takeaway:** The prompt is designed for governance. It tells a future LLM exactly what evidence to use and explicitly blocks unsupported claims.

## 7. Offline Rule-Based LLM-Style Generator

This function simulates the final text output without calling a paid model. It uses deterministic rules to produce fraud analyst style explanations from model outputs and SHAP drivers.

In [ ]:
def probability_band(probability, threshold):
    if probability >= 0.85:
        return "very high"
    if probability >= threshold:
        return "elevated"
    if probability >= threshold * 0.75:
        return "near-threshold"
    if probability >= 0.10:
        return "low-to-moderate"
    return "low"


def format_driver_sentence(drivers, direction):
    if not drivers:
        return f"No material {direction} SHAP drivers were provided for this example."

    pieces = []
    for driver in drivers[:4]:
        pieces.append(
            f"{driver['description']} ({driver['feature']}, SHAP {driver['shap_value']:+.4f})"
        )
    return "; ".join(pieces) + "."


def suggested_action(record):
    probability = record["fraud_probability"]
    threshold = record["decision_threshold"]
    case_type = record["case_type"]

    if record["prediction_label"] == "Fraud" and probability >= 0.85:
        return "Prioritize manual review, verify transaction context, and consider temporary hold or step-up authentication according to policy."
    if record["prediction_label"] == "Fraud":
        return "Send to analyst review queue and compare the SHAP drivers with customer history and recent activity."
    if case_type == "false negative":
        return "Review as a missed fraud example; consider whether threshold tuning, new features, or secondary rules could improve capture."
    return "No immediate fraud action is suggested by the model; continue routine monitoring unless external risk signals appear."


def generate_offline_fraud_explanation(record):
    probability = record["fraud_probability"]
    threshold = record["decision_threshold"]
    band = probability_band(probability, threshold)
    above_below = "above" if probability >= threshold else "below"

    risk_summary = (
        f"Transaction {record['transaction_id']} was classified as {record['prediction_label']} "
        f"with a fraud probability of {probability:.3f}, which is {above_below} the "
        f"{threshold:.2f} model decision threshold. This places the case in a {band} risk band."
    )

    if record.get("transaction_amount") is not None:
        risk_summary += (
            f" The available amount fields show transaction_amount={record['transaction_amount']} "
            f"and log_amount={record['log_amount']}."
        )

    key_model_drivers = (
        "Positive SHAP drivers that increased the fraud score: "
        + format_driver_sentence(record["top_positive_shap_features"], "positive")
        + " Negative SHAP drivers that reduced the fraud score: "
        + format_driver_sentence(record["top_negative_shap_features"], "negative")
    )

    caveat = (
        "This explanation is grounded in model outputs and SHAP feature contributions. "
        "Features V1-V28 are anonymized PCA transaction behavior components, so the system does not assign them specific business meanings. "
        "The explanation should support analyst review rather than replace investigation judgment."
    )

    return {
        "transaction_id": record["transaction_id"],
        "case_type": record["case_type"],
        "prediction_label": record["prediction_label"],
        "actual_label_for_demo": record["actual_label_for_demo"],
        "fraud_probability": probability,
        "decision_threshold": threshold,
        "risk_summary": risk_summary,
        "key_model_drivers": key_model_drivers,
        "suggested_investigation_action": suggested_action(record),
        "confidence_caveat": caveat,
        "full_explanation": "\n".join([
            "Risk summary: " + risk_summary,
            "Key model drivers: " + key_model_drivers,
            "Suggested investigation action: " + suggested_action(record),
            "Confidence caveat: " + caveat,
        ]),
    }

**Recruiter-friendly takeaway:** This is an offline stand-in for an LLM response. The output structure matches what a fraud analyst would need: risk summary, drivers, recommended action, and caveat.

## 8. Generate Fraud Investigation Explanations

The generator creates explanations for high-risk fraud predictions, false negatives, and legitimate low-risk transactions.

In [ ]:
explanations = [generate_offline_fraud_explanation(record) for record in example_records]
explanations_df = pd.DataFrame(explanations)

display(explanations_df[[
    "transaction_id",
    "case_type",
    "prediction_label",
    "fraud_probability",
    "risk_summary",
    "suggested_investigation_action",
]])

**Recruiter-friendly takeaway:** The same explanation framework handles flagged fraud, missed fraud, and low-risk legitimate cases, which is useful for both operational review and model improvement.

## 9. Review Analyst-Style Outputs

Each explanation is formatted with the sections a fraud analyst would expect.

In [ ]:
for explanation in explanations:
    print("=" * 100)
    print(explanation["transaction_id"], "|", explanation["case_type"])
    print("-" * 100)
    print(explanation["full_explanation"])
    print()

**Recruiter-friendly takeaway:** These outputs demonstrate how a technical model can become a usable fraud investigation assistant while staying grounded in evidence.

## 10. Save Generated Explanations

The CSV output can be used by a dashboard, Streamlit prototype, or manual review workflow.

In [ ]:
explanations_df.to_csv(EXPLANATIONS_PATH, index=False)
print(f"Saved explanations to: {EXPLANATIONS_PATH}")
display(explanations_df.head())

## 11. Save Markdown Explanation Report

The Markdown report provides a recruiter-friendly artifact that summarizes the prototype and includes example explanations.

In [ ]:
def build_markdown_report(explanations_df):
    lines = [
        "# LLM Fraud Explanation Layer Prototype",
        "",
        "This report was generated by `04_llm_fraud_explanation_layer.ipynb`.",
        "",
        "## Purpose",
        "",
        "This offline prototype converts fraud model outputs and SHAP feature contributions into human-readable investigation explanations.",
        "No OpenAI API, LangChain call, or paid API is used in this notebook.",
        "",
        "## Model Context",
        "",
        f"- Final model: {MODEL_NAME}",
        f"- Feature version: {FEATURE_VERSION}",
        "- Approximate performance: PR-AUC 0.816, ROC-AUC 0.986, precision 0.924, recall 0.768, F1 0.839",
        f"- Decision threshold used in examples: {MODEL_THRESHOLD:.2f}",
        "",
        "## Grounding Rules",
        "",
        "- Explanations are grounded in model outputs and SHAP feature contributions.",
        "- V1-V28 are anonymized PCA transaction behavior components.",
        "- The explanation layer does not claim to know the real business meaning of anonymized PCA features.",
        "- Generated text should support analyst review, not replace human investigation judgment.",
        "",
        "## Example Explanations",
        "",
    ]

    for row in explanations_df.itertuples(index=False):
        lines.extend([
            f"### {row.transaction_id} - {row.case_type}",
            "",
            f"**Prediction:** {row.prediction_label}",
            f"**Fraud probability:** {row.fraud_probability:.3f}",
            "",
            f"**Risk summary:** {row.risk_summary}",
            "",
            f"**Key model drivers:** {row.key_model_drivers}",
            "",
            f"**Suggested investigation action:** {row.suggested_investigation_action}",
            "",
            f"**Confidence caveat:** {row.confidence_caveat}",
            "",
        ])

    lines.extend([
        "## Future Integration Path",
        "",
        "This offline prototype can later connect to the OpenAI API, LangChain, or a Streamlit app by replacing the rule-based generator with a controlled LLM call.",
        "The same structured prompt and grounding rules should be preserved so the LLM converts SHAP outputs into readable explanations without inventing unsupported reasons.",
    ])
    return "\n".join(lines)


report_text = build_markdown_report(explanations_df)
REPORT_PATH.write_text(report_text, encoding="utf-8")
print(f"Saved Markdown report to: {REPORT_PATH}")
print(report_text[:1200])

## 12. Future Integration with OpenAI API, LangChain, or Streamlit

This notebook is intentionally offline, but the architecture is ready for a production-style LLM layer.

A future implementation could:
- Use the trained Random Forest to score a transaction
- Use SHAP to calculate local feature contributions for that transaction
- Package the model score, threshold, and top positive/negative SHAP drivers into the structured prompt template
- Send that prompt to the OpenAI API or through a LangChain chain with strict grounding instructions
- Display the generated explanation in a Streamlit fraud review app

The key principle should remain the same: SHAP provides model-grounded reasons, and the LLM converts those SHAP outputs into human-readable fraud investigation explanations. The LLM should not invent explanations, assign business meanings to anonymized PCA features, or override the model evidence.